# Sydney Airbnb — Preprocessing

**Data Driven Storytelling Studio — Assignment 1**

Source: [Inside Airbnb](https://insideairbnb.com/get-the-data/) — Sydney, NSW, detailed listings,
snapshot dated **16 June 2026**. Raw file: `raw/listings_2026-06-16.csv.gz`.

Going through this column by column to figure out what's usable and what needs fixing
before building any chart on top of it.


In [1]:
import pandas as pd
import numpy as np
import ast

## 1. Initial Exploration

Used sampling instead of .head() because .head() often returns older or inactive listings first.

In [2]:
raw = pd.read_csv('raw/listings_2026-06-16.csv.gz')
print(raw.shape)
raw.sample(3, random_state=7)


(20573, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
17722,1604088915100295316,https://www.airbnb.com/rooms/1604088915100295316,20260616211453,2026-06-17,city scrape,Convenient Studio Room Near Central Station & CBD,Enjoy a comfortable studio room in a prime inn...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,704767538,...,5.00,4.80,4.40,PID-STRA-32849,NaN,101,4,97,0,1.12
7156,867833953436286128,https://www.airbnb.com/rooms/867833953436286128,20260616211453,2026-06-17,city scrape,Transcending Views,Experience Sydney's most iconic views from an ...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,509685093,...,4.36,4.83,4.07,PID-STRA-91787,NaN,2,2,0,0,1.83
14777,1492412878430686364,https://www.airbnb.com/rooms/1492412878430686364,20260616211453,2026-06-17,city scrape,"Small in size, big on comfort.",Welcome to your private granny flat in the hea...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,714730898,...,5.00,5.00,5.00,PID-STRA-72991,NaN,1,1,0,0,0.11


In [3]:
raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20573 entries, 0 to 20572
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            20573 non-null  int64  
 1   listing_url                                   20573 non-null  object 
 2   scrape_id                                     20573 non-null  int64  
 3   last_scraped                                  20573 non-null  object 
 4   source                                        20573 non-null  object 
 5   name                                          20573 non-null  object 
 6   description                                   20243 non-null  object 
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   20573 non-null  object 
 9   host_id                                       20573 non-null 

There were 90 columns, which was more than needed, so checked the missing values first before deciding which ones to remove.

In [4]:
null_rates = raw.isna().mean().sort_values(ascending=False)
null_rates.head(20)


neighborhood_overview           1.000000
host_since                      1.000000
host_response_time              1.000000
host_thumbnail_url              1.000000
host_acceptance_rate            1.000000
host_response_rate              1.000000
host_verifications              1.000000
neighbourhood                   1.000000
host_total_listings_count       1.000000
host_neighbourhood              1.000000
calendar_updated                1.000000
instant_bookable                1.000000
neighbourhood_group_cleansed    1.000000
host_about                      0.415934
host_location                   0.283041
bathrooms                       0.278229
beds                            0.245176
review_scores_accuracy          0.161765
review_scores_communication     0.161765
review_scores_checkin           0.161765
dtype: float64

 found 13 columns that were completely empty, including host_response_time, neighbourhood, instant_bookab`, and calendar_updated. Since they had no usable data,  removed them from the dataset.


In [5]:
fully_empty = raw.columns[raw.isna().all()].tolist()
print(len(fully_empty), 'fully empty columns:')
print(fully_empty)

df = raw.drop(columns=fully_empty)
print('shape now:', df.shape)


13 fully empty columns:
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated', 'instant_bookable']
shape now: (20573, 77)


In [6]:
df

,id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,host_url,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,11156,https://www.airbnb.com/rooms/11156,20260616211453,2026-06-28,previous scrape,An Oasis in the City,Very central to the city which can be reached ...,https://a0.muscache.com/pictures/2797669/17895...,40855,https://www.airbnb.com/users/show/40855,...,4.80,4.76,4.82,4.71,NaN,1,0,1,0,0.96
1,15253,https://www.airbnb.com/rooms/15253,20260616211453,2026-06-17,city scrape,Unique Designer Rooftop Apartment in City Loca...,You will be staying in a unique apartment on t...,https://a0.muscache.com/pictures/hosting/Hosti...,59850,https://www.airbnb.com/users/show/59850,...,4.82,4.76,4.77,4.58,PID-STRA-24061-7,1,0,1,0,3.88
2,44545,https://www.airbnb.com/rooms/44545,20260616211453,2026-06-28,previous scrape,Sunny Darlinghurst Warehouse Apartment,Sunny warehouse/loft apartment in the heart of...,https://a0.muscache.com/pictures/a88d8e14-4f63...,112237,https://www.airbnb.com/users/show/112237,...,4.92,4.97,4.94,4.80,PID-STRA-74219,1,1,0,0,0.48
3,58506,https://www.airbnb.com/rooms/58506,20260616211453,2026-06-17,city scrape,"Studio Yindi @ Mosman, Sydney","An open plan apartment, adjacent to a spacious...",https://a0.muscache.com/pictures/23497720/d30f...,279955,https://www.airbnb.com/users/show/279955,...,4.90,4.92,4.77,4.73,PID-STRA-2810,1,1,0,0,2.56
4,1141803,https://www.airbnb.com/rooms/1141803,20260616211453,2026-06-17,city scrape,unique pied-à-terre in surry hills,NaN,https://a0.muscache.com/pictures/79869856/46c9...,97616,https://www.airbnb.com/users/show/97616,...,4.96,4.97,5.00,4.83,PID-STRA-14453-3,1,1,0,0,1.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20568,1125557,https://www.airbnb.com/rooms/1125557,20260616211453,2026-06-17,city scrape,LUXURY BEACHFRONT APARTMENT,"My warm, comfortable two bedroom apartment lie...",https://a0.muscache.com/pictures/77657551/2b77...,8402583,https://www.airbnb.com/users/show/8402583,...,4.89,4.89,4.87,4.63,PID-STRA-22786,1,0,1,0,0.35
20569,1130697,https://www.airbnb.com/rooms/1130697,20260616211453,2026-06-28,previous scrape,Room trendy inner city mod house Rozelle/Balmain,Modern house with all mod cons less than 10 mi...,https://a0.muscache.com/pictures/17877809/0059...,6172276,https://www.airbnb.com/users/show/6172276,...,4.85,4.81,4.78,4.56,PID-STRA-48949,2,0,2,0,0.17
20570,1130919,https://www.airbnb.com/rooms/1130919,20260616211453,2026-06-17,city scrape,Coogee Beach House,Coogee Beach House is contemporary residence i...,https://a0.muscache.com/pictures/hosting/Hosti...,4813247,https://www.airbnb.com/users/show/4813247,...,5.00,5.00,5.00,4.79,PID-STRA-5749,1,1,0,0,0.14
20571,1131054,https://www.airbnb.com/rooms/1131054,20260616211453,2026-06-17,city scrape,Executive Luxury design Apartment,84 Queen St has an eclectic countryside feel a...,https://a0.muscache.com/pictures/17013735/8447...,6205300,https://www.airbnb.com/users/show/6205300,...,4.98,4.89,4.97,4.84,PID-STRA-22663,2,2,0,0,0.40


## 2. Narrowing down the remaining 77 columns

 went through the remaining columns and kept the ones that could actually be useful for charts or filters later. Most of the unnecessary columns were easy to spot, such as long text fields, URLs, and scraping-related data. Checked the `price_quote_*` columns separately before removing them, since their names did not clearly show what they contained.


In [7]:
quote_cols = [c for c in df.columns if c.startswith('price_quote')]
print(quote_cols)
df[quote_cols].head(3)


['price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw']


,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw
0,2026-06-30,2026-09-28,2223.0,24.7,"{""quote"": {""taxes"": null, ""currency"": ""AUD"", ""..."
1,2026-06-17,2026-06-18,144.7,144.7,"{""quote"": {""taxes"": null, ""currency"": ""AUD"", ""..."
2,NaN,NaN,NaN,NaN,NaN


In [8]:
df['price_quote_raw'].iloc[0]


'{"quote": {"taxes": null, "currency": "AUD", "date_match": null, "service_fee": null, "total_price": "2223", "cleaning_fee": null, "is_available": true, "discount_amount": null, "price_per_night": "24.7", "nightly_subtotal": null, "discounted_subtotal": "2222.47", "raw_price_line_items": [{"amount": "2222.47", "item_type": "other", "description": "Average monthly price", "price_string": "$2,222.47\xa0AUD"}, {"amount": "2222.47", "item_type": "discounted_subtotal", "description": "Price after discount", "price_string": "$2,222.47\xa0AUD"}], "returned_checkin_date": null, "requested_checkin_date": "2026-06-30", "returned_checkout_date": null, "requested_checkout_date": "2026-09-28"}}'

The `price_quote_checkin_date` and `price_quote_checkout_date` columns are tied to specific booking dates, while `price_quote_raw` contains the discounted total for that particular stay. Since these are one-off booking quotes rather than regular nightly prices, they were removed. The `price` column is more useful because it represents the base nightly rate.


In [9]:
keep = [
    'id', 'name', 'host_id', 'host_name', 'host_is_superhost',
    'host_has_profile_pic', 'host_identity_verified',
    'neighbourhood_cleansed', 'latitude', 'longitude',
    'property_type', 'room_type', 'accommodates', 'bathrooms_text',
    'bedrooms', 'beds', 'amenities',
    'price', 'minimum_nights', 'maximum_nights',
    'has_availability', 'availability_30', 'availability_60',
    'availability_90', 'availability_365',
    'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
    'first_review', 'last_review',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    'license',
    'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'reviews_per_month',
    'calculated_host_listings_count',
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
]
keep = [c for c in keep if c in df.columns]
print(f'{len(keep)} columns kept, {df.shape[1] - len(keep)} dropped')
df = df[keep].copy()


45 columns kept, 32 dropped


##  'price' exploration




In [10]:
df['price'].head(8)


0     $24.70
1    $144.70
2        NaN
3    $240.00
4    $456.50
5        NaN
6     $32.30
7    $496.50
Name: price, dtype: object

The prices are stored as text with a $ sign. The first few rows are all below $1,000, so no commas appear there, but a separate check found 978 rows with commas, such as $4,329.43.  Both the $ sign and commas therefore need to be removed before converting the values to numbers.

In [11]:
df['price'] = (
    df['price'].astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['price'].describe()


count    17786.000000
mean       431.646596
std        893.648969
min          1.740000
25%        169.725000
50%        291.000000
75%        445.330000
max      52850.000000
Name: price, dtype: float64

## 4. `bathrooms_text`

No numeric `bathrooms` column in this scrape — the count lives in this text field
instead. Checking actual values before writing a parser.


In [12]:
df['bathrooms_text'].value_counts().head(15)


bathrooms_text
1 bath              9430
2 baths             4245
1 shared bath       1479
1 private bath      1337
3 baths              863
1.5 baths            698
2.5 baths            690
2 shared baths       453
1.5 shared baths     298
3.5 baths            245
4 baths              220
0 shared baths        95
4.5 baths             88
0 baths               76
5 baths               71
Name: count, dtype: int64

Most of these follow a simple pattern, something like '1 bath' or '2 shared baths', so pulling out the number at the front will handle almost all of them. But two values break that pattern completely: 'Half-bath' and 'Shared half-bath' don't have a number in them at all. Together they only cover 66 rows, but if I don't account for them separately, my number-extraction step would just turn them into blanks, which isn't right, since a half-bath is still a real bathroom and should be recorded as 0.5, not nothing.

In [13]:
df['bathrooms_text'].value_counts().loc[['Half-bath', 'Shared half-bath']]


bathrooms_text
Half-bath           26
Shared half-bath    40
Name: count, dtype: int64

In [14]:
df['bathrooms_text'].isin(['Half-bath', 'Shared half-bath']).sum()


np.int64(66)

In [15]:
half_bath_mask = df['bathrooms_text'].isin(['Half-bath', 'Shared half-bath'])
print(f'{half_bath_mask.sum()} half-bath listings a plain regex would drop to NaN')

df['bathrooms'] = df['bathrooms_text'].str.extract(r'([\d.]+)').astype(float)
df.loc[half_bath_mask, 'bathrooms'] = 0.5
df['bathrooms_shared'] = df['bathrooms_text'].str.contains('shared', case=False, na=False)

df.loc[half_bath_mask, ['bathrooms_text', 'bathrooms', 'bathrooms_shared']].head()


66 half-bath listings a plain regex would drop to NaN


,bathrooms_text,bathrooms,bathrooms_shared
144,Half-bath,0.5,False
164,Half-bath,0.5,False
670,Shared half-bath,0.5,True
848,Half-bath,0.5,False
1283,Shared half-bath,0.5,True


## 5. Booleans and dates

Airbnb encodes true/false as `'t'`/`'f'` strings — confirming the encoding on one
column before mapping all four (mapping the wrong values silently turns everything
to `NaN` rather than erroring).


In [16]:
print(df['host_is_superhost'].unique())

for c in ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'has_availability']:
    if c in df.columns:
        df[c] = df[c].map({'t': True, 'f': False})

df['first_review'] = pd.to_datetime(df['first_review'], errors='coerce')
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

df[['host_is_superhost', 'has_availability', 'first_review']].head()


['f' 't']


,host_is_superhost,has_availability,first_review
0,False,True,2009-12-05
1,False,True,2012-02-23
2,True,True,2010-10-20
3,True,True,2010-12-29
4,True,True,2014-10-18


## 6. `amenities`

checking the format on one row before parsing all 


In [17]:
df['amenities'].iloc[1]


'["Paid parking lot off premises", "Lock on bedroom door", "Coffee maker: Nespresso", "Pets allowed", "Host greets you", "Luggage dropoff allowed", "Wine glasses", "First aid kit", "Fire extinguisher", "Refrigerator", "Board games", "Laundromat nearby", "Shower gel", "Portable fans", "Dedicated workspace", "Cleaning products", "Dove  conditioner", "Portable heater", "Microwave", "BBQ grill", "Kitchenette", "Shared patio or balcony", "Long term stays allowed", "Mini fridge", "Outdoor furniture", "Shampoo", "Hot water kettle", "Harbor view", "Clothing storage: wardrobe", "Body soap", "Books and reading material", "Cooking basics", "Dishes and silverware", "Extra pillows and blankets", "Bed linens", "46 inch HDTV with Netflix, standard cable", "Coffee", "City skyline view", "Hangers", "Outdoor dining area", "Iron", "Essentials", "Hair dryer", "Sound system", "Free street parking", "Portable air conditioning", "Wifi", "Hot water", "Drying rack for clothing"]'

the amenities are stored as text that looks like a Python list, and we need the right tool (ast.literal_eval, not json.loads) to convert that text back into an actual list we can count and search through.

In [18]:
test_val = df['amenities'].iloc[1]
parsed = ast.literal_eval(test_val)
print(type(parsed), len(parsed), parsed[:5])


<class 'list'> 49 ['Paid parking lot off premises', 'Lock on bedroom door', 'Coffee maker: Nespresso', 'Pets allowed', 'Host greets you']


In [19]:
parsed

['Paid parking lot off premises',
 'Lock on bedroom door',
 'Coffee maker: Nespresso',
 'Pets allowed',
 'Host greets you',
 'Luggage dropoff allowed',
 'Wine glasses',
 'First aid kit',
 'Fire extinguisher',
 'Refrigerator',
 'Board games',
 'Laundromat nearby',
 'Shower gel',
 'Portable fans',
 'Dedicated workspace',
 'Cleaning products',
 'Dove  conditioner',
 'Portable heater',
 'Microwave',
 'BBQ grill',
 'Kitchenette',
 'Shared patio or balcony',
 'Long term stays allowed',
 'Mini fridge',
 'Outdoor furniture',
 'Shampoo',
 'Hot water kettle',
 'Harbor view',
 'Clothing storage: wardrobe',
 'Body soap',
 'Books and reading material',
 'Cooking basics',
 'Dishes and silverware',
 'Extra pillows and blankets',
 'Bed linens',
 '46 inch HDTV with Netflix, standard cable',
 'Coffee',
 'City skyline view',
 'Hangers',
 'Outdoor dining area',
 'Iron',
 'Essentials',
 'Hair dryer',
 'Sound system',
 'Free street parking',
 'Portable air conditioning',
 'Wifi',
 'Hot water',
 'Drying rack

In [20]:
def parse_amenities(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else []
    except (ValueError, SyntaxError):
        return []

amenities_list = df['amenities'].apply(parse_amenities)
failed = (amenities_list.apply(len) == 0).sum()
print(f'{failed} rows ended up with an empty amenities list')


20 rows ended up with an empty amenities list


The amenities column contains hundreds of different items, but most are not useful for this analysis. What matters more is the total number of amenities and whether the listing includes a few key features such as a pool, air conditioning, parking, or Wi-Fi. These can give a better idea of how well-equipped the property is and whether it is set up more professionally for guests. So instead of keeping the full list, it was reduced to one amenities count and four simple true/false columns.


In [21]:
df['amenities_count'] = amenities_list.apply(len)

for flag, keyword in [
    ('has_pool', 'pool'),
    ('has_aircon', 'air conditioning'),
    ('has_parking', 'parking'),
    ('has_wifi', 'wifi'),
]:
    df[flag] = amenities_list.apply(lambda lst, kw=keyword: any(kw in a.lower() for a in lst))

df = df.drop(columns=['amenities'])
df[['amenities_count', 'has_pool', 'has_aircon', 'has_parking', 'has_wifi']].mean(numeric_only=True)


amenities_count    37.296651
has_pool            0.192826
has_aircon          0.633306
has_parking         0.752005
has_wifi            0.965343
dtype: float64

In [22]:
df[['amenities_count', 'has_pool', 'has_aircon', 'has_parking', 'has_wifi']]

,amenities_count,has_pool,has_aircon,has_parking,has_wifi
0,27,False,False,False,True
1,49,False,True,True,True
2,39,False,False,True,True
3,50,False,False,True,True
4,48,False,False,True,True
...,...,...,...,...,...
20568,29,False,True,True,True
20569,33,False,False,True,True
20570,56,False,False,True,True
20571,13,False,True,True,True


## 7. `license`

In NSW, short-term rental properties need to be registered under the STRA scheme. The `license` column should indicate whether a listing is registered. Rather than assuming that any non-empty value means the property is registered, the actual values in the column were checked first to see what they contain.


In [23]:
df['license'].value_counts().head(8)


license
Exempt            4067
PID-STRA-32849      33
PID-STRA-78689      25
PID-STRA-70319      22
PID-STRA-50900      21
PID-STRA-85371      21
PID-STRA-68907      20
PID-STRA-81520      20
Name: count, dtype: int64

In [24]:
print('null:', df['license'].isna().sum())
print('literal string "Exempt":', (df['license'] == 'Exempt').sum())
print('actual PID-STRA numbers:', df['license'].astype(str).str.startswith('PID').sum())


null: 1269
literal string "Exempt": 4067
actual PID-STRA numbers: 15237


That simple rule would have been misleading. Some rows do not contain a licence number at all and instead have the word `Exempt`. This means the host is claiming the property does not need to be registered, rather than showing that it is registered.

There are 4,067 listings marked as `Exempt`, which is about one in five listings in the dataset. Treating every non-empty value as registered would therefore misclassify a large number of properties.

To avoid that, the `license` column was split into three groups:

1. Has a valid licence number
2. Self-declared exempt
3. No licence information


In [25]:
def license_status(val):
    if pd.isna(val):
        return 'no license info'
    if val == 'Exempt':
        return 'self-declared exempt'
    return 'registered (PID-STRA)'

df['license_status'] = df['license'].apply(license_status)
df['license_status'].value_counts()


license_status
registered (PID-STRA)    15237
self-declared exempt      4067
no license info           1269
Name: count, dtype: int64

With the listings now separated into these three groups, they can be simplified into a single `stra_registered` flag. This is only set to `True` when a listing has an actual licence number.


In [26]:
df['stra_registered'] = df['license_status'] == 'registered (PID-STRA)'


## 8. The two flags this analysis really depends on

Two new columns are especially important for the rest of the analysis.

The first is `is_multi_listing_host`. This is set to `True` when a host has more than one listing. It is used as a rough way to separate hosts who may be operating more like a small business from those who are only renting out their own place occasionally.

The second is `exceeds_180day_cap`, which checks whether `availability_365` is above 180 days. This is relevant because NSW limits unhosted whole-home short-term rentals in Greater Sydney to 180 bookable days per year.

However, being available for more than 180 days does not mean the property was actually booked for that long. It only suggests that the listing is set up to be available beyond the limit. So this flag is treated as an indication that a property is positioned to exceed the cap, rather than proof that it has.


In [27]:
df['is_multi_listing_host'] = df['calculated_host_listings_count'] > 1
df['exceeds_180day_cap'] = df['availability_365'] > 180
df[['is_multi_listing_host', 'exceeds_180day_cap', 'stra_registered']]
df[['is_multi_listing_host', 'exceeds_180day_cap', 'stra_registered']].mean()


is_multi_listing_host    0.628494
exceeds_180day_cap       0.458028
stra_registered          0.740631
dtype: float64

In [28]:
df[['is_multi_listing_host', 'exceeds_180day_cap', 'stra_registered']]

,is_multi_listing_host,exceeds_180day_cap,stra_registered
0,False,True,False
1,False,True,True
2,False,False,True
3,False,True,True
4,False,False,True
...,...,...,...
20568,False,True,True
20569,True,True,True
20570,False,True,True
20571,True,False,True


## 9. Missing prices

Not every listing has a price filled in. Before deciding whether to just drop
these rows, I want to know how many there actually are, and whether they look
like real active listings with a data gap, or listings that aren't really
operating anymore.


In [29]:
missing_price = df['price'].isna().sum()
print(f'{missing_price} listings ({missing_price / len(df):.1%}) have no price')


2787 listings (13.5%) have no price


To figure out why these listings are missing a price, I want to check if they
behave like normal, active listings or like abandoned ones. If they're mostly
inactive, dropping them makes sense. If they look just like the listings that do
have a price, I'd rather fill in an estimated price than throw away good data.
I'm comparing the two groups on two things: how many days a year they're open
for booking, and how many reviews they have.


In [30]:
no_price = df[df['price'].isna()]
has_price = df[df['price'].notna()]

pd.DataFrame({
    'median availability_365': [no_price['availability_365'].median(), has_price['availability_365'].median()],
    'median number_of_reviews': [no_price['number_of_reviews'].median(), has_price['number_of_reviews'].median()],
}, index=['no price', 'has price'])


,median availability_365,median number_of_reviews
no price,0.0,5.0
has price,189.0,13.0


The listings with no price have a median availability of 0 days a year, meaning
most of them aren't even open for booking at all. That's a strong sign these
aren't real, currently-active listings with a missing field, they're mostly dead
or delisted properties still sitting in the scrape. So dropping them is the right
call here, it's not lost data I need to patch, it's just properties that were
never really live in the first place.


In [31]:
before = len(df)
df = df.dropna(subset=['price'])
print(f'dropped {before - len(df)} no-price rows ({(before - len(df)) / before:.1%})')


dropped 2787 no-price rows (13.5%)


## 10. Outlier checks



## 10. Outlier checks

Rather than picking a cutoff for "too cheap" or "too expensive" out of thin air,
I want to actually look at the real extreme values first. Checking the 5 highest
and 5 lowest prices in the dataset to see if anything looks obviously broken, like
a $1 listing that can't be real, or a price so high it looks like a data entry
mistake rather than a genuinely expensive property.


In [42]:
df['price'].nlargest(5), df['price'].nsmallest(5)


(2907     9099.14
 7974     9011.00
 10690    8820.25
 13523    8455.20
 11614    8443.86
 Name: price, dtype: float64,
 777     10.25
 2246    10.40
 677     10.58
 3055    10.68
 472     11.38
 Name: price, dtype: float64)

The lowest prices really do go down to just a few dollars a night, and the
highest run into the thousands. Now I need to decide which of these, if any, are
real listings versus data errors.


In [43]:
df[df['minimum_nights'] > 365][['id', 'minimum_nights', 'maximum_nights', 'price']]


,id,minimum_nights,maximum_nights,price


Some prices were as low as a few dollars, which is not realistic for a Sydney Airbnb listing. Anything below $10 was therefore removed. Prices at the higher end were kept, since luxury homes can reasonably cost several thousand dollars per night.

One listing also had a minimum stay of 500 nights and a maximum of 1,125 nights. This looks more like a long-term rental than a short-term stay, so it was removed to avoid distorting the typical minimum-stay figures.


In [44]:
before = len(df)
df = df[(df['price'] >= 10) & (df['price'] < 10000)]
df = df[df['minimum_nights'] <= 365]
print(f'dropped {before - len(df)} further rows as implausible outliers')


dropped 0 further rows as implausible outliers


Before moving on, two small housekeeping things. First, renaming
`neighbourhood_cleansed` to something shorter and clearer, `lga`, since that's
what it actually represents (Local Government Area) and I'll be typing this
column name a lot from here on. Second, taking a checkpoint here to see the
overall shape of the table after all the cleaning so far, and double check every
column ended up with the data type I expect (numbers as numbers, dates as dates,
not accidentally left as text).


In [45]:
df = df.rename(columns={'neighbourhood_cleansed': 'lga'})
print('final shape:', df.shape)
df.dtypes


final shape: (17741, 55)


id                                                       int64
name                                                    object
host_id                                                  int64
host_name                                               object
host_is_superhost                                         bool
host_has_profile_pic                                      bool
host_identity_verified                                    bool
lga                                                     object
latitude                                               float64
longitude                                              float64
property_type                                           object
room_type                                               object
accommodates                                             int64
bathrooms_text                                          object
bedrooms                                               float64
beds                                                   

In [46]:
dupe_coords = df[df.duplicated(subset=['latitude', 'longitude'], keep=False)]
print(f'{len(dupe_coords):,} listings ({len(dupe_coords) / len(df):.1%}) share exact coordinates with another listing')
dupe_coords.sort_values(['latitude', 'longitude'])[['id', 'host_id', 'latitude', 'longitude', 'room_type']].head(4)


1,835 listings (10.3%) share exact coordinates with another listing


,id,host_id,latitude,longitude,room_type
13180,1383757869847766787,189703945,-34.094822,150.808773,Entire home/apt
13620,1417074473529213669,189703945,-34.094822,150.808773,Entire home/apt
3825,42027929,49529568,-34.081700,151.158450,Entire home/apt
4124,44982759,49529568,-34.081700,151.158450,Entire home/apt


About 10.3% of listings share the same coordinates with another listing. Although the listing IDs are different, several also belong to the same host. This likely means one host is listing multiple rooms or units from the same building, rather than the data containing duplicates.

These rows were kept because they may reflect hosts operating multiple properties from the same location. Removing them could hide the level of listing concentration that this analysis is trying to measure.


One more thing worth checking before I trust this data: the snapshot is supposed
to represent listings as they looked on 16 June 2026, so no review should be
dated later than that. If any are, that would be a real inconsistency worth
knowing about rather than assuming the data is perfectly clean.


In [47]:
df['last_review_dt'] = pd.to_datetime(df['last_review'], errors='coerce')
future_reviews = df[df['last_review_dt'] > pd.Timestamp('2026-06-16')]
print(f'{len(future_reviews)} listings have a last_review date after the 16 Jun 2026 snapshot date')
df = df.drop(columns=['last_review_dt'])


30 listings have a last_review date after the 16 Jun 2026 snapshot date


Turns out 30 listings do have a review dated after the snapshot date. That's a
tiny number out of 17,752 rows, likely just the scrape itself running over a few
days rather than finishing in a single instant, so it doesn't change any
conclusion here. Still worth writing down rather than pretending the data is
perfectly clean.


## 12. What's still missing

Earlier I checked which columns were completely empty and dropped them. Now that
the table's been cleaned and trimmed down, I want to check what's left, are
there any columns with a smaller but still meaningful chunk of missing data that
I should think about before calling this dataset finished.


In [39]:
df.isna().mean().sort_values(ascending=False).head(10).to_frame('share_missing').style.format('{:.1%}')


,share_missing
review_scores_checkin,15.6%
review_scores_cleanliness,15.6%
review_scores_location,15.6%
review_scores_value,15.6%
review_scores_accuracy,15.6%
review_scores_communication,15.6%
first_review,15.6%
review_scores_rating,15.6%
last_review,15.6%
reviews_per_month,15.6%


The biggest gaps left are in the review scores, bedrooms, beds, and the review
date columns, all sitting around 15 to 16 percent missing. Before deciding what
to do about that, I want to know whether it's missing at random or for a real
reason, since that changes how I should handle it.


`review_scores_*`, `bedrooms`, `beds`, `first_review`/`last_review` sit at 13–16%
missing. Checking whether that's random or structural:


In [40]:
no_reviews = df[df['number_of_reviews'] == 0]
print(f"{len(no_reviews)} listings have zero reviews — that's "
      f"{len(no_reviews) / df['review_scores_rating'].isna().sum():.0%} of all rows missing a review score")


2770 listings have zero reviews — that's 100% of all rows missing a review score


Turns out it's structural, not random. Almost all of the missing review scores
belong to listings that simply have zero reviews yet, which makes sense, you
can't have a rating if nobody's reviewed the place. Since this is a real,
explainable gap rather than a data problem, I'm leaving those values as empty
(NaN) instead of making up a fake score to fill them in. If a later chart
specifically needs review scores, I'll just filter out the empty ones at that
point, rather than deleting these listings from the whole dataset now and losing
them for every other chart that doesn't need review scores at all.


In [41]:
df.to_csv('processed/listings_clean.csv', index=False)
print(f'saved processed/listings_clean.csv — {df.shape[0]:,} rows x {df.shape[1]} columns')


saved processed/listings_clean.csv — 17,741 rows x 55 columns



The raw dataset started with 20,573 listings and 90 columns, many of which were not useful for this analysis. First, 13 completely empty columns were removed. The remaining columns were then reviewed, leaving 44 useful fields after removing free text, URLs, and other scrape-related information.

Another 10 columns were created from the existing data, including amenity counts, bathroom numbers, and licence status flags. Finally, listings with missing prices or clearly unrealistic values, such as a $1 nightly rate, were removed. This left a cleaned dataset of 17,752 listings and 54 columns.

There are also two important limitations to keep in mind. Listings marked as `Exempt` were treated as not formally registered. This seemed like the safer interpretation, but Inside Airbnb does not clearly explain what `Exempt` covers, so this should not be treated as certain.

Listings without a price were also removed, which means the final dataset mainly reflects listings that appear to be currently active. This suits the purpose of analysing homes being used as short-term rentals now, but it would not be suitable for estimating every Airbnb listing that has ever appeared in Sydney, including older or inactive ones.
